In [1]:
import pandas as pd
import xgboost as xgb
import mlflow
import mlflow.xgboost
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

mlflow.set_experiment("BEAM")

<Experiment: artifact_location='file:///C:/Users/nilab/Documents/artin/mlruns/1', creation_time=1772527214054, experiment_id='1', last_update_time=1772527214054, lifecycle_stage='active', name='BEAM', tags={}, workspace='default'>

In [2]:
df = pd.read_csv(r"C:\Users\nilab\Documents\artin\ENB2012_data.csv")

X = df.drop(columns=['Y1', 'Y2'])
y = df[['Y1', 'Y2']] 

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [4]:
with mlflow.start_run(run_name="XGBoost_Fixed_Baseline"):
    params = {
        "objective": "reg:squarederror",
        "n_estimators": 100,
        "learning_rate": 0.1,
        "max_depth": 5,
        "random_state": 42
    }
    
    mlflow.log_params(params)
    
    model = xgb.XGBRegressor(**params)
    model.fit(X_train, y_train)
    
    predictions = model.predict(X_test)
    mse = mean_squared_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)
    
    mlflow.log_metric("mse", mse)
    mlflow.log_metric("r2_score", r2)
    
    # Updated line to remove the warning
    mlflow.xgboost.log_model(model, name="model")
    
    print(f"Success! Model trained and logged to MLflow.")
    print(f"R2 Score: {r2:.4f}")

Success! Model trained and logged to MLflow.
R2 Score: 0.9922


In [7]:
# List of depths to test
depths = [3, 5, 8]

for depth in depths:
    run_name = f"XGBoost_Depth_{depth}"
    
    with mlflow.start_run(run_name=run_name):
        params = {
            "objective": "reg:squarederror",
            "n_estimators": 100,
            "learning_rate": 0.1,
            "max_depth": depth,     
            "random_state": 42
        }
        
        mlflow.log_params(params)
        
        model = xgb.XGBRegressor(**params)
        model.fit(X_train, y_train)
        
        predictions = model.predict(X_test)
        mse = mean_squared_error(y_test, predictions)
        r2 = r2_score(y_test, predictions)
        
        mlflow.log_metric("mse", mse)
        mlflow.log_metric("r2_score", r2)
        
        mlflow.xgboost.log_model(model, name="model")
        
        print(f"Finished Run: {run_name} | R2 Score: {r2:.4f}")

Finished Run: XGBoost_Depth_3 | R2 Score: 0.9853
Finished Run: XGBoost_Depth_5 | R2 Score: 0.9922
Finished Run: XGBoost_Depth_8 | R2 Score: 0.9877


In [6]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

experiment = client.get_experiment_by_name("BEAM")
runs = client.search_runs(experiment_ids=[experiment.experiment_id])

latest_run = runs[0]
print(f"Run Name: {latest_run.data.tags['mlflow.runName']}")
print(f"R2 Score: {latest_run.data.metrics['r2_score']:.4f}")

Run Name: XGBoost_Fixed_Baseline
R2 Score: 0.9922
